# Play — VinRobotics mjlab (Google Colab)

Notebook chuyển từ `scripts/play.py`. Trên Colab (không có màn hình) viewer native không mở được,
nên ta **quay video** (`video=True`) rồi hiển thị trong notebook, hoặc dùng viewer `viser` qua web.


## 1. Cài đặt & lấy source

In [ ]:
import os

REPO_URL = ""  # ví dụ: https://github.com/<org>/vinrobotics_mjlab.git
REPO_DIR = "/content/vinrobotics_mjlab"

if REPO_URL and not os.path.isdir(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

for cand in (REPO_DIR, os.getcwd()):
    if os.path.isfile(os.path.join(cand, "setup.py")):
        os.chdir(cand)
        break
print("cwd:", os.getcwd())

In [ ]:
# !pip install -q mjlab==1.4.0 mujoco==3.8.1 mujoco-warp==3.8.1 warp-lang==1.13.0
# !pip install -q -e .

## 2. Môi trường render

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())
os.environ["MUJOCO_GL"] = "egl"
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

## 3. Chọn task

In [ ]:
import mjlab.tasks  # noqa: F401
import src.tasks    # noqa: F401
from mjlab.tasks.registry import list_tasks

for t in list_tasks():
    print("  -", t)

TASK_ID = "VR-M3-1-12DOF-Flat"

## 4. Cấu hình play

- `agent="zero"` / `"random"`: xem robot không cần checkpoint (kiểm tra scene nhanh).
- `agent="trained"`: cần `checkpoint_file` trỏ tới `model_*.pt` từ notebook train.
- `viewer="viser"`: mở viewer web (Colab headless không dùng được `native`).


In [ ]:
from scripts.play import PlayConfig
import dataclasses

cfg = PlayConfig(
    agent="zero",                 # "zero" | "random" | "trained"
    checkpoint_file=None,          # ví dụ: "logs/rsl_rl/.../model_1000.pt" khi agent="trained"
    num_envs=1,
    video=True,                    # quay video thay vì viewer cửa sổ
    video_length=300,
    viewer="viser",
    no_terminations=True,
)
print(cfg)

## 5. Chạy play

Với `video=True` và `agent="trained"`, video được lưu trong `logs/.../videos/play/`.
Dummy agent (`zero`/`random`) không hỗ trợ video — dùng `viser` để xem hoặc chuyển sang `trained`.


In [ ]:
from scripts.play import run_play

run_play(TASK_ID, cfg)

## 6. Hiển thị video (nếu đã quay với agent="trained")

In [ ]:
import glob
from IPython.display import Video, display

vids = sorted(glob.glob("logs/rsl_rl/**/videos/play/*.mp4", recursive=True))
if vids:
    print("Video mới nhất:", vids[-1])
    display(Video(vids[-1], embed=True, width=480))
else:
    print("Chưa có video. Chạy với agent='trained' và video=True.")